# CNN-Based Malaria Parasite Recognition in Human Blood Smear Images


### Setup — Colab (Team / TA) or Local

**Running on Colab (recommended for GPU):**
1. Notebook toolbar → **Select Kernel → Colab** → sign in
2. Set **Hardware accelerator = GPU** (T4 if available)
3. Run cells top to bottom — repo and data download automatically

**Running locally (VS Code):**
- Just run cells top to bottom — setup detects local mode automatically


In [16]:
import sys, os, torch
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists("/content/COMP-472"):
        os.system("git clone -q https://github.com/iamBolu/COMP-472 /content/COMP-472")
    DRIVE_PROJECT_ROOT = "/content/COMP-472"
    os.system("pip install -q -r /content/COMP-472/requirements.txt")

    # Data lives on Google Drive — MyDrive/COMP 472/Project/data/raw
    DATA_ROOT = Path("/content/drive/MyDrive/COMP 472/Project/data/raw")
    IN_COLAB  = True
except ImportError:
    DRIVE_PROJECT_ROOT = str(Path(os.getcwd()).parent) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    DATA_ROOT = Path(DRIVE_PROJECT_ROOT) / "data" / "raw"
    IN_COLAB  = False

sys.path.insert(0, DRIVE_PROJECT_ROOT)
os.chdir(DRIVE_PROJECT_ROOT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Runtime : {'Google Colab' if IN_COLAB else 'VS Code (local)'}")
print(f"Device  : {device} {'← GPU ready' if device.type == 'cuda' else '← no GPU detected'}")
print(f"Code dir: {DRIVE_PROJECT_ROOT}")
print(f"Data dir: {DATA_ROOT}")
print(f"Data exists: {DATA_ROOT.exists()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime : Google Colab
Device  : cuda ← GPU ready
Code dir: /content/COMP-472
Data dir: /content/drive/MyDrive/COMP 472/Project/data/raw
Data exists: True


---
## Member 1: Data Visualization & Reporting

### Phase A (do **first**, in parallel with Member 2)
- Bar chart: class distribution per dataset
- Image grid: random samples per class
- Summary table: dataset sizes, class counts, imbalance
- **Define plotting functions:**
  1. `plot_training_curves(history, title)` — expects a `history` dict with keys: `"train_loss"`, `"val_loss"`, `"train_acc"`, `"val_acc"` (all lists, one value per epoch)
  2. `plot_confusion_matrix(labels, preds, class_names, title)` — expects `labels` and `preds` as numpy arrays, `class_names` as a list
  3. `print_classification_report(labels, preds, class_names)` — expects `labels` and `preds` as numpy arrays, `class_names` as a list

### Phase B (do **last**, after Members 3, 4, 5 finish training)
- For each model/dataset, call the plotting functions above using the results provided by Members 3, 4, 5.

---

### **IMPORTANT: Output Format Contract for Members 3, 4, 5**

To ensure compatibility, all training results **must** be provided in the following format:

#### For each architecture (ResNet-18, VGG-16, MobileNet-V2) and each dataset:
```python
# Example for ResNet-18 (Member 3)
resnet_history = {}  # resnet_history["malaria"] = history dict (see below)
resnet_labels  = {}  # resnet_labels["malaria"]  = numpy array (test split)
resnet_preds   = {}  # resnet_preds["malaria"]   = numpy array (test split)

# Example for VGG-16 (Member 4)
vgg_history = {}
vgg_labels  = {}
vgg_preds   = {}

# Example for MobileNet-V2 (Member 5)
mobilenet_history = {}
mobilenet_labels  = {}
mobilenet_preds   = {}
```

#### The `history` dict **must** have:
```python
history = {
    "train_loss": [...],  # float per epoch
    "val_loss":   [...],
    "train_acc":  [...],  # float 0–1 per epoch
    "val_acc":    [...],
}
```

#### For Transfer Learning (Member 5, one dataset only):
```python
tl_freeze_history,  tl_freeze_labels,  tl_freeze_preds   = {...}, {...}, {...}
tl_finetune_history, tl_finetune_labels, tl_finetune_preds = {...}, {...}, {...}
```
- All dicts are keyed by dataset name (e.g. `"malaria"`)
- All arrays are numpy arrays (test split only)

**Member 1 will only accept results in this format.**

---

### Example usage in Member 1 Phase B
```python
# For each model/dataset:
plot_training_curves(resnet_history["malaria"], "ResNet-18 Malaria")
plot_confusion_matrix(resnet_labels["malaria"], resnet_preds["malaria"], class_names, "ResNet-18 Malaria Confusion Matrix")
print_classification_report(resnet_labels["malaria"], resnet_preds["malaria"], class_names)
# ...repeat for VGG-16, MobileNet-V2, TL runs
```
---


---
## Member 2: Pipeline Implementation

**Goal:** Data quality checks, common transform policy, and DataLoader creation for all 3 datasets.  
All 9 baseline training runs (ResNet-18, VGG-16, MobileNet-V2 × miracle9to9, orvile, malaria) consume `dataloaders_registry` directly.


### Step 1 - Imports & Constants


In [17]:
import pandas as pd
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# DATA_ROOT and device are set in cell 3 based on runtime
IMAGE_INPUT_SIZE = 128
BATCH_SIZE       = 32
NUM_WORKERS      = 0
DATASET_NAMES    = ["miracle9to9", "malaria", "iml_malaria"]
IMAGE_EXTENSIONS = {".jpg", ".png"}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Exists:    {DATA_ROOT.exists()}")


DATA_ROOT: /content/drive/MyDrive/COMP 472/Project/data/raw
Exists:    True


### Step 2 - Data Inventory

Scan all images under `data/raw/` and report class distribution and imbalance ratios per dataset/split.
PIL validation is skipped — `broken_files_report.txt` (notebooks/) was generated in a prior run.


In [18]:
all_image_paths = pd.Series([
    path for path in DATA_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
])

if all_image_paths.empty:
    raise FileNotFoundError(f"No images found under '{DATA_ROOT.resolve()}'. Expected: data/raw/<dataset>/<split>/<class>/image.ext")

valid_images_inventory_df = pd.DataFrame({
    "dataset":   all_image_paths.map(lambda p: p.relative_to(DATA_ROOT).parts[0]),
    "split":     all_image_paths.map(lambda p: p.relative_to(DATA_ROOT).parts[1]),
    "class_raw": all_image_paths.map(lambda p: p.parent.name),
    "filepath":  all_image_paths,
})
print(f"Total images: {len(valid_images_inventory_df)}")

class_distribution_df = (
    valid_images_inventory_df
    .groupby(["dataset", "split", "class_raw"])
    .size()
    .reset_index(name="image_count")
)
print("\nClass distribution:")
print(class_distribution_df.to_string(index=False))

split_totals_df = (
    valid_images_inventory_df
    .groupby(["dataset", "split"])
    .size()
    .reset_index(name="total_images")
)
print("\nSplit totals:")
print(split_totals_df.to_string(index=False))

imbalance_df = (
    class_distribution_df
    .groupby(["dataset", "split"])["image_count"]
    .agg(max_count="max", min_count="min")
    .assign(imbalance_ratio=lambda df: (df["max_count"] / df["min_count"]).round(2))
    .reset_index()
)
print("\nImbalance ratios (max / min class count):")
print(imbalance_df.to_string(index=False))


Total images: 44365

Class distribution:
    dataset split   class_raw  image_count
iml_malaria  test  gametocyte           13
iml_malaria  test        ring            7
iml_malaria  test    schizont            2
iml_malaria  test trophozoite            3
iml_malaria train  gametocyte           81
iml_malaria train        ring           41
iml_malaria train    schizont           11
iml_malaria train trophozoite           21
iml_malaria   val  gametocyte           16
iml_malaria   val        ring            8
iml_malaria   val    schizont            2
iml_malaria   val trophozoite            4
    malaria  test  gametocyte            3
    malaria  test        ring           45
    malaria  test    schizont            8
    malaria  test trophozoite           38
    malaria train  gametocyte           49
    malaria train        ring          108
    malaria train    schizont           88
    malaria train trophozoite          326
    malaria   val  gametocyte            4
    malaria  

### Step 3 - Transform Policy

One shared policy for all 9 baseline training runs.  
- `training_transforms` : resize → geometric augmentation → color jitter → normalize  
- `evaluation_transforms` : resize → normalize only (no augmentation on val/test)


In [19]:
training_transforms = transforms.Compose([
    transforms.Resize((IMAGE_INPUT_SIZE, IMAGE_INPUT_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(90),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

evaluation_transforms = transforms.Compose([
    transforms.Resize((IMAGE_INPUT_SIZE, IMAGE_INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("training_transforms :")
print(training_transforms)
print("\nevaluation_transforms :")
print(evaluation_transforms)


training_transforms :
Compose(
    Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-90.0, 90.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

evaluation_transforms :
Compose(
    Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


### Step 4 - MalariaDataset

Custom `Dataset` that works for all three datasets.  
Accepts a pre-filtered `split_inventory_df` slice from `valid_images_inventory_df` (Step 2), so only validated images are ever loaded.  
Handles `orvile`'s duplicate training folders (`"gametocyte 2"` → `"gametocyte"`) by stripping the trailing numeric suffix via vectorized regex on class folder names.


In [20]:
class MalariaDataset(Dataset):
    def __init__(self, split_inventory_df, transform=None):
        normalized_class_names = split_inventory_df["class_raw"].str.replace(r"\s+\d+$", "", regex=True)

        unique_classes   = sorted(normalized_class_names.unique())
        class_to_index   = {class_name: index for index, class_name in enumerate(unique_classes)}

        self.image_paths  = split_inventory_df["filepath"].to_numpy()
        self.labels       = normalized_class_names.map(class_to_index).to_numpy()
        self.class_names  = unique_classes
        self.class_to_idx = class_to_index
        self.transform    = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[index])


### Step 5 - Datasets & DataLoaders

`datasets_registry[dataset_name][split]` → `MalariaDataset`  
`dataloaders_registry[dataset_name][split]` → `DataLoader` ready for training  

All 9 scratch models and 2 transfer learning models consume `dataloaders_registry`.


In [21]:
SPLIT_TRANSFORMS = {"train": training_transforms, "val": evaluation_transforms, "test": evaluation_transforms}

datasets_registry = {
    dataset_name: {
        split: MalariaDataset(
            valid_images_inventory_df[
                (valid_images_inventory_df["dataset"] == dataset_name) &
                (valid_images_inventory_df["split"]   == split)
            ].reset_index(drop=True),
            transform=split_transform,
        )
        for split, split_transform in SPLIT_TRANSFORMS.items()
    }
    for dataset_name in DATASET_NAMES
}

dataloaders_registry = {
    dataset_name: {
        split: DataLoader(
            dataset,
            batch_size  = BATCH_SIZE,
            shuffle     = (split == "train"),
            num_workers = NUM_WORKERS,
            pin_memory  = torch.cuda.is_available(),
            drop_last   = (split == "train"),
        )
        for split, dataset in split_datasets.items()
        if len(dataset) > 0
    }
    for dataset_name, split_datasets in datasets_registry.items()
}

for dataset_name, split_datasets in datasets_registry.items():
    for split_name, dataset in split_datasets.items():
        if len(dataset) == 0:
            print(f"{dataset_name:15s} | {split_name:5s} | SKIPPED since 0 valid images")
            continue
        num_batches = len(dataloaders_registry[dataset_name][split_name])
        print(f"{dataset_name:15s} | {split_name:5s} | {len(dataset):5d} images | {num_batches:3d} batches | classes: {dataset.class_names}")


miracle9to9     | train | 23425 images | 732 batches | classes: ['Parasitized', 'Uninfected']
miracle9to9     | val   |  4133 images | 130 batches | classes: ['Parasitized', 'Uninfected']
miracle9to9     | test  | 15832 images | 495 batches | classes: ['Parasitized', 'Uninfected']
malaria         | train |   571 images |  17 batches | classes: ['gametocyte', 'ring', 'schizont', 'trophozoite']
malaria         | val   |   101 images |   4 batches | classes: ['gametocyte', 'ring', 'schizont', 'trophozoite']
malaria         | test  |    94 images |   3 batches | classes: ['gametocyte', 'ring', 'schizont', 'trophozoite']
iml_malaria     | train |   154 images |   4 batches | classes: ['gametocyte', 'ring', 'schizont', 'trophozoite']
iml_malaria     | val   |    30 images |   1 batches | classes: ['gametocyte', 'ring', 'schizont', 'trophozoite']
iml_malaria     | test  |    25 images |   1 batches | classes: ['gametocyte', 'ring', 'schizont', 'trophozoite']


---
## Member 2: New Coding Tasks

### New Coding Tasks (TO DO)
1. **Reproducibility** — `torch.manual_seed(42)`, `torch.backends.cudnn.deterministic = True`, `torch.backends.cudnn.benchmark = False`.
2. **`build_model(arch, num_classes)`** — returns ResNet-18, VGG-16, or MobileNet-V2 with `weights=None` and final layer replaced. Called by all members.
3. **`count_parameters(model)`** — returns total trainable parameter count. Print a comparison table for all 3 architectures (`num_classes=4`).
4. **`train_one_epoch(model, loader, criterion, optimizer, device)`** — one forward + backward pass; returns average loss and accuracy.
5. **`evaluate(model, loader, criterion, device)`** — one forward-only pass; returns average loss and accuracy.

### Hints
- Final layer per arch: `resnet.fc`, `vgg.classifier[6]`, `mobilenet.classifier[1]` → **Lab_Tutorial_Week_8 (1).ipynb, line 152**
- `count_parameters` → **Lab_Tutorial_Week_8 (1).ipynb, line 54**
- `train_one_epoch` → **Lab_Tutorial_Week_9.ipynb, line 1000**
- `evaluate` → **Lab_Tutorial_Week_9.ipynb, line 1042**
- Use `torchvision.models` with `weights=None`. No TensorFlow / Keras.

### Requirements Alignment
> *"The architectures should vary in terms of complexity (number of layers and learnable parameters)."* — Project Requirements

The parameter count table directly justifies the complexity difference between ResNet-18, VGG-16, and MobileNet-V2.

### Tasks for Final Report & Video
- **Methodology > Models**: parameter count table; one-line description per architecture.


---
## Member 3: Training Loop + ResNet-18

### Uses from Member 2 *(must be defined before running this section)*
- `build_model(arch, num_classes)` — instantiate ResNet-18 with `weights=None`
- `train_one_epoch(model, loader, criterion, optimizer, device)`
- `evaluate(model, loader, criterion, device)`

### New Coding Tasks (TO DO)
1. **`collect_predictions(model, loader, device)`** — returns `(labels, preds)` as **numpy arrays** over the full loader.
2. **`fit(model, train_loader, val_loader, optimizer, criterion, device, epochs, checkpoint_path)`** — calls `train_one_epoch` + `evaluate` each epoch, saves best-val checkpoint, returns a `history` dict.
3. **Train ResNet-18 × 3** (miracle9to9, malaria, iml_malaria): `lr=0.001`, Adam, CrossEntropyLoss, `epochs=10`. Save checkpoints to `Path(DRIVE_PROJECT_ROOT) / "checkpoints"`.

### Output contract for Member 1 Phase B
`fit()` must return a dict with **exactly** these keys:
```python
history = {
    "train_loss": [...],  # float per epoch
    "val_loss":   [...],
    "train_acc":  [...],  # float 0–1 per epoch
    "val_acc":    [...],
}
```
After training, run `collect_predictions` on the **test** split and store results — Members 4 & 5 follow the same pattern:
```python
resnet_history = {}  # resnet_history["miracle9to9"] = history dict
resnet_labels  = {}  # resnet_labels["miracle9to9"]  = numpy array
resnet_preds   = {}  # resnet_preds["miracle9to9"]   = numpy array
```
Member 1's `plot_training_curves(history, title)`, `plot_confusion_matrix(labels, preds, class_names, title)`, and `print_classification_report(labels, preds, class_names)` will consume these directly.

### Hints
- `collect_predictions` → **Lab_Tutorial_Week_9.ipynb, line 1064**
- `fit` with history + checkpointing → **Lab_Tutorial_Week_9.ipynb, line 1136**
- `num_classes = len(datasets_registry[dataset_name]["train"].class_names)`
- No TensorFlow / Keras.

### Requirements Alignment
> *"train 9 different models... for fair comparison, all models should be trained under the same conditions (same hyperparameters)."* — Project Requirements

### Tasks for Final Report & Video
- Report val and test accuracy for each of the 3 ResNet-18 runs.


---
## Member 4: VGG-16 + Optimization

### Uses from Member 2 & 3 *(must be defined before running this section)*
- `build_model("vgg16", num_classes)` — from Member 2
- `train_one_epoch`, `evaluate` — from Member 2
- `fit(model, train_loader, val_loader, optimizer, criterion, device, epochs, checkpoint_path)` — from Member 3
- `collect_predictions(model, loader, device)` — from Member 3

### New Coding Tasks (TO DO)
1. **Train VGG-16 × 3** (miracle9to9, malaria, iml_malaria): `lr=0.001`, Adam, CrossEntropyLoss, `epochs=10`.
2. **LR sweep** — on one dataset, train VGG-16 with `lr ∈ {0.01, 0.001, 0.0001}`. Log val accuracy per LR. Pick the best and explain why.

### Output contract for Member 1 Phase B
Follow the **same storage pattern** as Member 3:
```python
vgg_history = {}  # vgg_history["miracle9to9"] = history dict with train_loss/val_loss/train_acc/val_acc
vgg_labels  = {}  # vgg_labels["miracle9to9"]  = numpy array (test split)
vgg_preds   = {}  # vgg_preds["miracle9to9"]   = numpy array
```
Member 1's `plot_training_curves`, `plot_confusion_matrix`, and `print_classification_report` will consume these directly.

### Hints
- `build_model("vgg16", num_classes)` → **Lab_Tutorial_Week_8 (1).ipynb, line 152**
- LR sweep loop → **Lab_Tutorial_Week_9.ipynb, line ~1136**
- No TensorFlow / Keras.

### Requirements Alignment
> *"Each team should choose at least one of the 9 models trained from scratch and attempt to optimize it through hyperparameter tuning... only from: learning rate, batch size, and loss function."* — Project Requirements

### Tasks for Final Report & Video
- Report val and test accuracy for each of the 3 VGG-16 runs.
- Table: LR → val accuracy; justify the best LR choice.


---
## Member 5: MobileNet-V2 + Transfer Learning + T-SNE

### Uses from Member 2 & 3 *(must be defined before running this section)*
- `build_model("mobilenet_v2", num_classes)` — from Member 2
- `train_one_epoch`, `evaluate` — from Member 2
- `fit(model, train_loader, val_loader, optimizer, criterion, device, epochs, checkpoint_path)` — from Member 3
- `collect_predictions(model, loader, device)` — from Member 3

### New Coding Tasks (TO DO)
1. **Train MobileNet-V2 × 3** (miracle9to9, malaria, iml_malaria): `lr=0.001`, Adam, CrossEntropyLoss, `epochs=10`.
2. **TL Run 1** (pick one dataset) — pretrained weights (`weights="IMAGENET1K_V1"`), freeze backbone, train head only for 10 epochs.
3. **TL Run 2** (same dataset) — unfreeze full network, fine-tune for 10 epochs from the frozen checkpoint.
4. **T-SNE** — for ≥4 models: replace classifier with `nn.Identity()`, extract test-set embeddings, `sklearn.manifold.TSNE(n_components=2)`, scatter plot coloured by class label.
5. **Final comparison table** — val accuracy, test accuracy, macro-F1 for all 11 models as a sorted DataFrame.

### Output contract for Member 1 Phase B
Follow the **same storage pattern** as Members 3 & 4:
```python
mobilenet_history  = {}  # mobilenet_history["miracle9to9"]  = history dict with train_loss/val_loss/train_acc/val_acc
mobilenet_labels   = {}  # mobilenet_labels["miracle9to9"]   = numpy array (test split)
mobilenet_preds    = {}  # mobilenet_preds["miracle9to9"]    = numpy array

# TL runs — store under the chosen dataset name with a descriptive key
tl_freeze_history,  tl_freeze_labels,  tl_freeze_preds   = {...}, {...}, {...}
tl_finetune_history, tl_finetune_labels, tl_finetune_preds = {...}, {...}, {...}
```
Member 1's `plot_training_curves`, `plot_confusion_matrix`, `print_classification_report` will consume all of these.

### Hints
- `build_model("mobilenet_v2", num_classes)` → **Lab_Tutorial_Week_8 (1).ipynb, line 152**
- `build_transfer_model()` (freeze backbone) → **Lab_Tutorial_Week_9.ipynb, line 7004**
- `fit_transfer_model()` → **Lab_Tutorial_Week_9.ipynb, line 7035**
- F1: `sklearn.metrics.f1_score(labels, preds, average="macro")`
- No TensorFlow / Keras.

### Requirements Alignment
> *"2 additional models using Transfer Learning (any 2 combinations of dataset+architecture)."* — Project Requirements  
> *"Use TSNE or Grad-CAM to visualize the performance of at least 4 different models."* — Project Requirements

### Tasks for Final Report & Video
- Compare scratch vs TL-freeze vs TL-fine-tune — table + discussion.
- T-SNE scatter plots in **Results & Discussion**.
- **Conclusion** — best model per dataset, key findings, limitations.
